# Append web-search blocked domains with API Management

This lab shows how to add organization domains to a Responses API request while preserving the caller's existing blocklist. APIM only applies the merge when a `web_search` tool is present.

For developers familiar with APIM and Foundry. You need an existing APIM service, a Foundry backend or resource endpoint, an existing model deployment that supports `web_search`, Azure CLI authentication, and the repository's Python environment (`uv sync` at the repo root).

1. Load existing resource settings from `.env`.
2. Inspect the policy and prepare routing to the existing backend.
3. Deploy the lab API.
4. Send requests with and without web search and inspect the backend request in APIM tracing.

See [README.md](README.md) for configuration and policy behavior, and [Microsoft's domain-filtering documentation](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search#domain-filtering) for the request format.

## 1. Load configuration

Run with this lab directory as the working directory. The loader reads the root `.env`, then this lab's `.env`; process variables win. To select another existing file, set `env_file` below or use `LAB_ENV_FILE`.

`BACKEND_ID` / `APIM_BACKEND_ID` reuses an existing APIM Foundry backend and takes precedence over `AZURE_OPENAI_ENDPOINT`. A resource endpoint creates only a backend on existing APIM. Neither option provisions Foundry or a model deployment. Avoid printing configuration dictionaries because they can contain keys.

In [ ]:
from pathlib import Path
from src.lab import LAB_DIR, load_config, prepare_deployment, deploy, send_response
from dotenv import load_dotenv

load_dotenv()
# Example: env_file = "../secure-responses-api/.env"
env_file = None
config = load_config(env_file)
model = config.get("AZURE_OPENAI_DEPLOYMENT") or config.get("AZURE_OPENAI_DEPLOYMENT_NAME")
if not model:
    raise ValueError("Set AZURE_OPENAI_DEPLOYMENT to an existing deployment supporting web_search.")
print("Configuration loaded; model deployment:", model)

## 2. Inspect the policy and existing backend

The first `<choose>` in [policy.xml](policy.xml) checks `tools` using a preserved copy of the request body. Its branch validates the web-search filters and appends missing domains, preserving existing entries, `allowed_domains`, and unrelated fields.

`web_search_preview` does not support this filtering contract and is left unchanged. Use `web_search` here.

The next cell performs read-only Azure lookups. It fails if a configured resource has been deleted. For backend pools or custom URLs, supply `BACKEND_RESPONSES_PATH` as described in the README.

In [ ]:
from src.lab import load_config, prepare_deployment

# Reload configuration if this cell is run directly after a kernel restart.
if "config" not in globals():
    config = load_config(globals().get("env_file"))

prepared = prepare_deployment(config)
print("APIM service:", prepared["parameters"]["apimServiceName"])
print("Existing backend:", prepared["parameters"]["backendId"] or "Create lab backend for configured endpoint")
print("Backend Responses path:", prepared["parameters"]["backendResponsesPath"])
print("Lab Responses URL:", prepared["responses_url"])

## 3. Deploy the dedicated lab API

This cell creates or updates the lab API and policy through [main.bicep](main.bicep). Choose an unused `APIM_API_NAME` and `APIM_API_PATH` for the first run. It reuses the APIM service and the selected backend; an endpoint-only configuration creates a lab backend.

The APIM subscription key used below must have access to the new API (all APIs or this API's scope). The deployment helper enables APIM's system-assigned managed identity if needed and grants it **Cognitive Services OpenAI User** on the Foundry account. This applies to reused backends and to endpoint configurations without an API key. Existing identities and role assignments are preserved. The API policy obtains a Microsoft Entra token for `https://ai.azure.com` and sends it in the backend `Authorization` header.

The signed-in Azure CLI account needs permission to create role assignments on Foundry, such as Owner or Role Based Access Control Administrator. The helper discovers the account from the backend hostname in the APIM subscription; set `AZURE_OPENAI_RESOURCE_ID` for an account in another subscription or with a custom hostname. For pools, use a comma-separated list in `AZURE_OPENAI_RESOURCE_IDS`. New grants can take a few minutes to propagate.

In [ ]:
deployment = deploy(config, prepared)
print("Deployment state:", deployment["properties"]["provisioningState"])
print("Responses URL:", prepared["responses_url"])

## 4. Control request: no web-search tool

This makes a model request through APIM with no `tools` property. Blocklist validation and `set-body` are skipped. In the portal Test tab, enable tracing and verify that the backend request has no added `tools` or `filters` properties.

In [ ]:
def show_text(response):
    for item in response.get("output", []):
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                print(content["text"])

without_web_search = {"model": model, "input": "Reply with the word hello.", "store": False}
control_response = send_response(config, prepared, without_web_search)
show_text(control_response)

## 5. Web search without an existing blocklist

Only the tool declaration is sent by the client. APIM adds `filters.blocked_domains` with the 11 organization domains. `tool_choice` requests a web-search call, and `include` asks Foundry to return consulted sources.

In [ ]:
with_web_search = {
    "model": model,
    "input": "Search for Azure API Management announcements and summarize one with a source link.",
    "tools": [{"type": "web_search"}],
    "tool_choice": {"type": "web_search"},
    "include": ["web_search_call.action.sources"],
    "store": False,
}
search_response = send_response(config, prepared, with_web_search)
show_text(search_response)

## 6. Preserve the caller's blocklist and other filters

The client blocks `example.com` and `youtube.com`. The gateway must preserve `example.com`, keep a single `youtube.com`, and append the other 10 domains. It must also retain the caller's `allowed_domains` and `search_context_size`.

Run this request, then repeat it in APIM's portal Test tab with tracing enabled. Check the **Backend request body**, rather than relying on the response to echo tool settings. The expected merged array is in the README.

In [ ]:
from copy import deepcopy

with_existing_blocklist = deepcopy(with_web_search)
with_existing_blocklist["tools"][0].update({
    "search_context_size": "low",
    "filters": {
        "allowed_domains": ["learn.microsoft.com", "azure.microsoft.com"],
        "blocked_domains": ["example.com", "youtube.com"],
    },
})
merged_response = send_response(config, prepared, with_existing_blocklist)
show_text(merged_response)

# This is the client-side request. APIM adds domains only after receiving it.
assert with_existing_blocklist["tools"][0]["filters"]["blocked_domains"] == ["example.com", "youtube.com"]

## 7. Inspect returned search sources

Sources provide a useful observation of the search result. They do not prove that APIM transformed every request correctly; use the trace and local policy checks for that.

In [ ]:
for item in merged_response.get("output", []):
    if item.get("type") == "web_search_call":
        for source in item.get("action", {}).get("sources", []):
            print(source.get("url", ""))

## Exercise: case-insensitive duplicate handling

Change the caller's `youtube.com` entry to `YouTube.COM` and add `example.org`. Predict the backend blocklist, then use APIM tracing to check it.

Expected: both caller-only domains survive, `YouTube.COM` retains its spelling and position, and APIM does not append a second `youtube.com`. Existing duplicate entries are preserved, while the policy introduces no new duplicates.

Optional extension: add a function tool alongside `web_search` and verify its JSON remains unchanged. A common mistake is putting `blocked_domains` directly on the tool; it belongs inside `filters`.

In [ ]:
exercise_request = deepcopy(with_existing_blocklist)
exercise_request["tools"][0]["filters"]["blocked_domains"] = ["example.com", "YouTube.COM", "example.org"]
# Send after predicting the result:
# exercise_response = send_response(config, prepared, exercise_request)

## Local verification and cleanup

From a terminal in this directory, run `python -m unittest discover -s tests -v`. Policy checks compile the actual expressions from `policy.xml` with .NET 8; they require NuGet access on first run. Build the infrastructure with `az bicep build --file main.bicep --outfile /tmp/web-search-blocklist.json`.

When finished, use [clean-up-resources.ipynb](clean-up-resources.ipynb) to delete the dedicated lab API and any lab-created backend. Keep the shared APIM and Foundry resources.